# Gemma 2 2B IT fine-tuning with Unsloth and TRL

This notebook fine-tunes Gemma 2 2B IT on a French educational dialogue dataset. It prepares the JSONL files, applies the Gemma chat format, evaluates against validation examples, and saves the resulting LoRA adapter.

The training split contains 40 examples, the validation split contains 5 examples, and the held-out test split contains 10 examples. Training uses only the training split; validation and test data remain separate.

The resulting adapter represents a French educational tutor style with progressive explanations, examples, and respectful corrections.

In [ ]:
# Install required packages
%pip install -q -U unsloth trl datasets peft accelerate bitsandbytes

## Hugging Face authentication

Hugging Face authentication provides access to the Gemma model repository.

In [ ]:
from huggingface_hub import login

login()

## Dataset files and output storage

The dataset files are selected from the local `datasets` directory:

- `train.jsonl` contains the training examples.
- `validation.jsonl` contains the evaluation examples.
- `test.jsonl` contains the held-out test examples.

Google Drive provides persistent storage for the adapter after the Colab runtime ends.

In [ ]:
from google.colab import drive, files

drive.mount('/content/drive')

uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

required_files = {'train.jsonl', 'validation.jsonl', 'test.jsonl'}
missing_files = required_files.difference(uploaded.keys())
if missing_files:
    raise FileNotFoundError(f'Missing required files: {sorted(missing_files)}')

In [ ]:
from datasets import load_dataset

# Load the uploaded files. Keep these filenames unchanged.
train_path = '/content/train.jsonl'
validation_path = '/content/validation.jsonl'
test_path = '/content/test.jsonl'

raw_dataset = load_dataset('json', data_files=train_path, split='train')
raw_validation = load_dataset('json', data_files=validation_path, split='train')
raw_test = load_dataset('json', data_files=test_path, split='train')
print('train:', raw_dataset)
print('validation:', raw_validation)
print('test:', raw_test)
print(raw_dataset[0])

## Dataset normalization

The source records use Gemma's `user` and `model` roles. The tutor description is included at the beginning of each first user message because the Gemma 2 chat template does not define a separate `system` role.

In [ ]:
SYSTEM_PROMPT = (
    "Tu es un professeur généraliste expert, patient et bienveillant. "
    "Explique progressivement, définis les termes importants, donne un exemple concret, "
    "corrige les erreurs sans jugement et adapte la longueur au niveau de l'apprenant. "
    "Pose une question de vérification lorsque c'est pertinent."
)

def normalize_messages(example):
    messages = []

    for index, msg in enumerate(example['messages']):
        role = msg['role']
        content = msg['content']
        if index == 0 and role == 'user':
            content = SYSTEM_PROMPT + '\n\nQuestion de l’apprenant :\n' + content

        messages.append({
            'role': role,
            'content': content,
        })

    return {'messages': messages}

raw_dataset = raw_dataset.map(normalize_messages)
raw_validation = raw_validation.map(normalize_messages)
raw_test = raw_test.map(normalize_messages)
print(raw_dataset[0])

In [ ]:
# Apply chat template using Gemma's tokenizer.
# This turns each multi-turn chat into a single training text string.

from unsloth import FastLanguageModel

model_name = 'google/gemma-2-2b-it'
max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

def format_conversation(example):
    return {
        'text': tokenizer.apply_chat_template(
            example['messages'],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

def make_text_dataset(dataset):
    return dataset.map(
        format_conversation,
        remove_columns=dataset.column_names,
    )

train_dataset = make_text_dataset(raw_dataset)
validation_dataset = make_text_dataset(raw_validation)
test_dataset = make_text_dataset(raw_test)

print(train_dataset[0]['text'])
print('Prepared examples:', len(train_dataset), len(validation_dataset), len(test_dataset))

## LoRA adapter configuration

The base model weights remain frozen while the LoRA matrices are trained. This reduces the memory required for the T4 runtime.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj',
        'k_proj',
        'v_proj',
        'o_proj',
        'gate_proj',
        'up_proj',
        'down_proj',
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

## Supervised fine-tuning

The training configuration uses one epoch, a batch size of two, gradient accumulation, and a maximum sequence length of 512 tokens.

In [ ]:
import torch
from trl import SFTConfig, SFTTrainer

output_dir = '/content/gemma-2-2b-it-lora-project'

training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    logging_steps=1,
    save_strategy='epoch',
    eval_strategy='epoch',
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_length=max_seq_length,
    dataset_text_field='text',
    packing=False,
    report_to='none',
    seed=3407,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    args=training_args,
)

trainer.train()

## Adapter export


In [ ]:
adapter_dir = '/content/drive/MyDrive/gemma-2-2b-it-lora-final'
import os
os.makedirs(adapter_dir, exist_ok=True)
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'Saved adapter to {adapter_dir}')

## Inference

The trained adapter generates a response to a French educational question.

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

messages = [{
    'role': 'user',
    'content': "Qu'est-ce qu'une variable en programmation ? Donne une explication simple."
}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
).to('cuda')

with torch.inference_mode():
    generated = model.generate(
        input_ids=inputs,
        max_new_tokens=180,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        use_cache=True,
    )

new_tokens = generated[0, inputs.shape[1]:]
output = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
print(output)

## Held-out test data

The test split remains separate from training and validation. Its formatted records are available for subsequent manual evaluation.

In [ ]:
print(test_dataset[0]['text'])
print(f'Test examples available for later evaluation: {len(test_dataset)}')

## Direct comparison: original model and fine-tuned model

This cell uses the current fine-tuned model first, then releases it and loads the original Gemma model. Both models receive the same prompts and generation settings. No adapter ZIP upload is required.

In [ ]:
import gc
import torch
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "Tu es un professeur généraliste expert, patient et bienveillant. "
    "Explique progressivement, définis les termes importants, donne un exemple concret, "
    "corrige les erreurs sans jugement et adapte la longueur au niveau de l'apprenant. "
    "Pose une question de vérification lorsque c'est pertinent."
)

prompts = [
    "Je ne comprends pas les fractions. Peux-tu m'aider progressivement ?",
    "Explique-moi la photosynthèse comme si j'avais dix ans.",
    "J'ai répondu que 2 + 3 × 4 = 20. Est-ce correct ?",
    "Qu'est-ce qu'une variable en programmation ? Donne une explication simple.",
    "Explique-moi le sujet.",
    "Explique la photosynthèse avec un peu de rigueur scientifique.",
    "Donne-moi un exercice simple sur les pourcentages, mais ne donne pas immédiatement la réponse.",
    "Peux-tu m'expliquer la différence entre un fait et une opinion avec des exemples ?",
    "Je n'ai pas réussi cet exercice, donc je suis mauvais en maths. Est-ce vrai ?",
    "Pourquoi une preuve par l'absurde fonctionne-t-elle ? Explique simplement.",
]

def make_inputs(current_tokenizer, question):
    messages = [{
        'role': 'user',
        'content': SYSTEM_PROMPT + '\n\nQuestion de l’apprenant :\n' + question,
    }]
    return current_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to('cuda')

def generate_comparison_answer(current_model, current_tokenizer, question):
    inputs = make_inputs(current_tokenizer, question)
    with torch.inference_mode():
        generated = current_model.generate(
            input_ids=inputs,
            max_new_tokens=300,
            do_sample=False,
            use_cache=True,
        )
    return current_tokenizer.decode(
        generated[0, inputs.shape[1]:],
        skip_special_tokens=True,
    ).strip()

# Generate the AFTER responses while the fine-tuned model is in memory.
after_answers = [
    generate_comparison_answer(model, tokenizer, question)
    for question in prompts
]

# Release the fine-tuned model before loading the original model on the T4.
del model
gc.collect()
torch.cuda.empty_cache()

# Load the original base model without the LoRA adapter.
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name='google/gemma-2-2b-it',
    max_seq_length=512,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(base_model)

# Generate the BEFORE responses with the same prompts and settings.
before_answers = [
    generate_comparison_answer(base_model, base_tokenizer, question)
    for question in prompts
]

for index, question in enumerate(prompts, start=1):
    print('\n' + '=' * 80)
    print(f'TEST {index}: {question}')
    print(f'\n--- BEFORE FINE-TUNING ---\n{before_answers[index - 1]}')
    print(f'\n--- AFTER FINE-TUNING ---\n{after_answers[index - 1]}')

## Prompt comparison using the current fine-tuned model

This section reuses the fine-tuned `model` and `tokenizer` already in the Colab runtime. It does not upload the adapter ZIP again.

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    "Tu es un professeur généraliste expert, patient et bienveillant. "
    "Explique progressivement, définis les termes importants, donne un exemple concret, "
    "corrige les erreurs sans jugement et adapte la longueur au niveau de l'apprenant. "
    "Pose une question de vérification lorsque c'est pertinent."
)

prompts = [
    "Je ne comprends pas les fractions. Peux-tu m'aider progressivement ?",
    "Explique-moi la photosynthèse comme si j'avais dix ans.",
    "J'ai répondu que 2 + 3 × 4 = 20. Est-ce correct ?",
    "Qu'est-ce qu'une variable en programmation ? Donne une explication simple.",
    "Explique-moi le sujet.",
    "Explique la photosynthèse avec un peu de rigueur scientifique.",
    "Donne-moi un exercice simple sur les pourcentages, mais ne donne pas immédiatement la réponse.",
    "Peux-tu m'expliquer la différence entre un fait et une opinion avec des exemples ?",
]

def generate_tutor_answer(question):
    messages = [{
        'role': 'user',
        'content': SYSTEM_PROMPT + '\n\nQuestion de l’apprenant :\n' + question,
    }]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to('cuda')

    with torch.inference_mode():
        generated = model.generate(
            input_ids=inputs,
            max_new_tokens=180,
            do_sample=False,
            use_cache=True,
        )

    return tokenizer.decode(
        generated[0, inputs.shape[1]:],
        skip_special_tokens=True,
    ).strip()

for index, question in enumerate(prompts, start=1):
    print('\n' + '=' * 80)
    print(f'TEST {index}: {question}')
    print(generate_tutor_answer(question))